# BÀI 1: HIỂU DỮ LIỆU, CHỌN DỮ LIỆU VÀ TIỀN XỬ LÝ
## Tập dữ liệu D2 - Online Retail

### 1. Đề xuất bộ dữ liệu và ma trận kỹ thuật

#### 1.1. Nguồn gốc, giấy phép, quy mô

- **Tên dataset:** Online Retail
- **Nguồn:** UCI Machine Learning Repository
- **Link:** https://archive.ics.uci.edu/dataset/352/online+retail
- **Tác giả:** Daqing Chen
- **Giấy phép:** CC BY 4.0
- **Nội dung:** Dữ liệu giao dịch bán lẻ trực tuyến.
- **Quy mô:** Được xác định trực tiếp bằng `df.shape`.

#### 1.2. Từ điển dữ liệu

| Thuộc tính | Ý nghĩa | Kiểu | Thang đo |
|---|---|---|---|
| `InvoiceNo` | Mã hóa đơn | Chuỗi | Định danh |
| `StockCode` | Mã sản phẩm | Chuỗi | Định danh |
| `Description` | Tên sản phẩm | Chuỗi | Danh nghĩa |
| `Quantity` | Số lượng sản phẩm | Số nguyên | Tỷ lệ |
| `InvoiceDate` | Ngày giờ giao dịch | DateTime | Thời gian |
| `UnitPrice` | Đơn giá sản phẩm | Số thực | Tỷ lệ |
| `CustomerID` | Mã khách hàng | Số | Định danh |
| `Country` | Quốc gia khách hàng | Chuỗi | Danh nghĩa |

#### 1.3. Tri thức lĩnh vực và đề xuất kỹ thuật

Dữ liệu ghi nhận các giao dịch mua hàng trực tuyến. Những sản phẩm cùng xuất hiện trong
một hóa đơn có thể phản ánh hành vi mua kèm của khách hàng.

- **Luật kết hợp:** Tìm các sản phẩm thường được mua cùng nhau.
- **Gom cụm:** Phân nhóm khách hàng theo hành vi mua hàng bằng RFM.

#### 1.4. Ma trận bộ dữ liệu × kỹ thuật

| Bộ dữ liệu | Phân lớp | Luật kết hợp | Gom cụm |
|---|:---:|:---:|:---:|
| **D2: Online Retail** |  | **X** | **X** |


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

RAW = Path("data/raw")
OUT = Path("outputs")
PROCESSED = Path("data/processed")

OUT.mkdir(exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

files = (
    list(RAW.glob("*.xlsx")) +
    list(RAW.glob("*.xls")) +
    list(RAW.glob("*.csv"))
)

if not files:
    raise FileNotFoundError("Không tìm thấy dữ liệu trong D2/data/raw")

file = files[0]

if file.suffix.lower() == ".csv":
    df = pd.read_csv(file, encoding="ISO-8859-1")
else:
    df = pd.read_excel(file)

print("Tệp dữ liệu:", file)
print("Kích thước ban đầu:", df.shape)
display(df.head())
display(df.dtypes)


### 2. Điều tra tiền xử lý

#### 2.1. Giá trị thiếu

`CustomerID` thiếu khi không xác định được khách hàng. Đây là thiếu có ý nghĩa nghiệp vụ,
không nên gán mã khách hàng giả. Các giao dịch này vẫn được giữ cho luật kết hợp nhưng
không dùng khi xây dựng RFM.

`Description` thiếu được thay bằng `StockCode` để bảo toàn thông tin sản phẩm.

In [ ]:
missing = pd.DataFrame({
    "Số lượng thiếu": df.isna().sum(),
    "Tỷ lệ thiếu (%)": (df.isna().mean() * 100).round(4)
})

display(missing[missing["Số lượng thiếu"] > 0])
missing.to_csv(
    OUT / "missing_report.csv",
    index=False,
    encoding="utf-8-sig"
)


#### 2.2. Nhiễu và ngoại lai

- Hóa đơn bắt đầu bằng `C` thường là giao dịch trả hàng hoặc hủy.
- `Quantity <= 0` không phải giao dịch mua hợp lệ.
- `UnitPrice <= 0` không tạo ra doanh thu hợp lệ.
- Ngoại lai được kiểm tra bằng Boxplot và quy tắc IQR.
- Các giao dịch không hợp lệ được loại bỏ để tránh làm sai doanh thu, luật kết hợp
  và kết quả gom cụm.

In [ ]:
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)
df["is_return"] = df["InvoiceNo"].str.startswith("C")

outlier_rows = []

for column in ["Quantity", "UnitPrice"]:
    values = pd.to_numeric(
        df[column],
        errors="coerce"
    ).dropna()

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_rows.append({
        "Thuộc tính": column,
        "Q1": q1,
        "Q3": q3,
        "Ngưỡng dưới": lower,
        "Ngưỡng trên": upper,
        "Số ngoại lai": int(
            ((values < lower) | (values > upper)).sum()
        )
    })

outliers = pd.DataFrame(outlier_rows)
display(outliers)

outliers.to_csv(
    OUT / "outlier_report.csv",
    index=False,
    encoding="utf-8-sig"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(x=df["Quantity"], ax=axes[0])
sns.boxplot(x=df["UnitPrice"], ax=axes[1])

axes[0].set_title("Quantity trước xử lý")
axes[1].set_title("UnitPrice trước xử lý")

plt.show()


#### 2.3. Thêm thuộc tính

Các thuộc tính mới được tạo để mô tả giao dịch, hóa đơn, khách hàng và sản phẩm:

- Thuộc tính thời gian: `InvoiceYear`, `InvoiceMonth`, `InvoiceDay`,
  `InvoiceHour`, `InvoiceWeekday`, `IsWeekend`.
- Thuộc tính giao dịch: `TotalPrice`, `ItemRevenueShare`.
- Thuộc tính hóa đơn: `InvoiceTotal`, `BasketSize`, `UniqueProducts`,
  `AvgUnitPrice`, `InvoiceItemCount`, `InvoiceAvgQuantity`,
  `InvoiceMaxQuantity`, `InvoiceMinQuantity`, `InvoiceStdQuantity`.
- Thuộc tính khách hàng: `CustomerTotalSpent`, `CustomerInvoiceCount`,
  `CustomerProductCount`, `CustomerAvgOrderValue`, `CustomerAvgQuantity`.
- Thuộc tính sản phẩm: `ProductTotalQuantity`, `ProductTotalRevenue`,
  `ProductInvoiceCount`, `ProductAvgPrice`.

Sau khi xử lý, dữ liệu `clean` có khoảng 35 thuộc tính.

In [ ]:
before = len(df)

clean = df[
    (~df["is_return"]) &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0) &
    (df["InvoiceNo"].notna()) &
    (df["StockCode"].notna())
].copy()

clean["Description"] = clean["Description"].fillna(
    clean["StockCode"].astype(str)
)

# Thuộc tính thời gian
clean["InvoiceYear"] = clean["InvoiceDate"].dt.year
clean["InvoiceMonth"] = clean["InvoiceDate"].dt.month
clean["InvoiceDay"] = clean["InvoiceDate"].dt.day
clean["InvoiceHour"] = clean["InvoiceDate"].dt.hour
clean["InvoiceWeekday"] = clean["InvoiceDate"].dt.weekday
clean["IsWeekend"] = (clean["InvoiceWeekday"] >= 5).astype(int)

# Doanh thu từng dòng
clean["TotalPrice"] = clean["Quantity"] * clean["UnitPrice"]

# Thuộc tính cấp hóa đơn
invoice_features = clean.groupby("InvoiceNo").agg(
    InvoiceTotal=("TotalPrice", "sum"),
    BasketSize=("Quantity", "sum"),
    UniqueProducts=("StockCode", "nunique"),
    AvgUnitPrice=("UnitPrice", "mean"),
    InvoiceItemCount=("StockCode", "count"),
    InvoiceAvgQuantity=("Quantity", "mean"),
    InvoiceMaxQuantity=("Quantity", "max"),
    InvoiceMinQuantity=("Quantity", "min"),
    InvoiceStdQuantity=("Quantity", "std")
).reset_index()

clean = clean.merge(
    invoice_features,
    on="InvoiceNo",
    how="left"
)

# Thuộc tính cấp khách hàng
customer_features = clean.dropna(
    subset=["CustomerID"]
).groupby("CustomerID").agg(
    CustomerTotalSpent=("TotalPrice", "sum"),
    CustomerInvoiceCount=("InvoiceNo", "nunique"),
    CustomerProductCount=("StockCode", "nunique"),
    CustomerAvgOrderValue=("InvoiceTotal", "mean"),
    CustomerAvgQuantity=("Quantity", "mean")
).reset_index()

clean = clean.merge(
    customer_features,
    on="CustomerID",
    how="left"
)

# Thuộc tính cấp sản phẩm
product_features = clean.groupby("StockCode").agg(
    ProductTotalQuantity=("Quantity", "sum"),
    ProductTotalRevenue=("TotalPrice", "sum"),
    ProductInvoiceCount=("InvoiceNo", "nunique"),
    ProductAvgPrice=("UnitPrice", "mean")
).reset_index()

clean = clean.merge(
    product_features,
    on="StockCode",
    how="left"
)

# Tỷ trọng doanh thu của sản phẩm trong hóa đơn
clean["ItemRevenueShare"] = (
    clean["TotalPrice"] / clean["InvoiceTotal"]
)

clean["InvoiceStdQuantity"] = clean["InvoiceStdQuantity"].fillna(0)

clean = clean.replace(
    [np.inf, -np.inf],
    np.nan
)

clean = clean.fillna(0)

print("Số dòng ban đầu:", before)
print("Số dòng sau xử lý:", len(clean))
print("Số dòng bị loại:", before - len(clean))
print("Số thuộc tính sau khi thêm:", clean.shape[1])

display(clean.head())
display(pd.DataFrame({
    "Tên thuộc tính": clean.columns
}))


### 3. Gom cụm khách hàng bằng RFM

RFM gồm:

- `Recency`: Số ngày từ lần mua gần nhất.
- `Frequency`: Số hóa đơn của khách hàng.
- `Monetary`: Tổng giá trị mua hàng.

Các thuộc tính RFM được log-transform và chuẩn hóa trước khi áp dụng K-Means.

In [ ]:
rfm_data = clean[
    clean["CustomerID"] != 0
].copy()

rfm_data = rfm_data.dropna(
    subset=["CustomerID", "InvoiceDate"]
)

snapshot = (
    rfm_data["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

rfm = rfm_data.groupby("CustomerID").agg(
    Recency=(
        "InvoiceDate",
        lambda values: (snapshot - values.max()).days
    ),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum")
).reset_index()

rfm_features = np.log1p(
    rfm[["Recency", "Frequency", "Monetary"]]
)

X = StandardScaler().fit_transform(rfm_features)

scores = []
labels_by_k = {}

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    scores.append({
        "Số cụm": k,
        "Silhouette Score": silhouette_score(X, labels)
    })

    labels_by_k[k] = labels

scores = pd.DataFrame(scores)

best_k = int(
    scores.loc[
        scores["Silhouette Score"].idxmax(),
        "Số cụm"
    ]
)

rfm["Cluster"] = labels_by_k[best_k]

print("Số cụm được chọn:", best_k)
display(scores)
display(
    rfm.groupby("Cluster")[
        ["Recency", "Frequency", "Monetary"]
    ].mean().round(2)
)

scores.to_csv(
    OUT / "clustering_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

rfm.to_csv(
    OUT / "rfm_with_clusters.csv",
    index=False,
    encoding="utf-8-sig"
)

sns.lineplot(
    data=scores,
    x="Số cụm",
    y="Silhouette Score",
    marker="o"
)

plt.title("Lựa chọn số cụm bằng Silhouette Score")
plt.show()


### 4. Luật kết hợp

Mỗi hóa đơn được biểu diễn thành một tập sản phẩm.

- **Support:** Tỷ lệ hóa đơn chứa tập sản phẩm.
- **Confidence:** Độ tin cậy của luật.
- **Lift:** Mức độ liên hệ giữa các sản phẩm.

In [ ]:
from mlxtend.frequent_patterns import (
    apriori,
    association_rules
)

basket = clean.pivot_table(
    index="InvoiceNo",
    columns="Description",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

top_products = basket.sum().nlargest(100).index
basket = basket[top_products]
basket = (basket > 0).astype(int)

frequent_itemsets = apriori(
    basket,
    min_support=0.01,
    use_colnames=True
)

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.25
)

rules = rules[
    rules["lift"] > 1
].sort_values(
    ["lift", "confidence"],
    ascending=False
)

display(frequent_itemsets.head())
display(rules.head(10))

frequent_itemsets.to_csv(
    OUT / "frequent_itemsets.csv",
    index=False,
    encoding="utf-8-sig"
)

rules.to_csv(
    OUT / "association_rules.csv",
    index=False,
    encoding="utf-8-sig"
)


### 5. Lưu dữ liệu đã tiền xử lý

Dữ liệu và các kết quả phân tích được lưu riêng trong thư mục D2.

In [ ]:
clean.to_csv(
    PROCESSED / "online_retail_processed.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu dữ liệu D2.")
print("Số thuộc tính cuối cùng:", clean.shape[1])


### 6. Sản phẩm nộp

- `D2/khao-sat-d2-hoan-chinh.ipynb`
- `D2/data/processed/online_retail_processed.csv`
- `D2/outputs/missing_report.csv`
- `D2/outputs/outlier_report.csv`
- `D2/outputs/rfm_with_clusters.csv`
- `D2/outputs/clustering_scores.csv`
- `D2/outputs/frequent_itemsets.csv`
- `D2/outputs/association_rules.csv`
